# Практическая работа 8. Задание 1
MST алгоритмами Краскала и Прима, разрез для ребра MST

In [1]:
import heapq

names = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
# ребра (вес, u, v), A=0, B=1, ..., H=7
# вес D-E на рисунке закрыт подписью ребра B-F, взял 5
edges = [
    (6, 0, 1), (4, 0, 2), (8, 0, 3),   # A-B, A-C, A-D
    (2, 1, 2), (7, 1, 4), (5, 1, 5),   # B-C, B-E, B-F
    (3, 2, 3),                         # C-D
    (5, 3, 4), (5, 3, 5), (6, 3, 6),   # D-E, D-F, D-G
    (4, 4, 5), (3, 4, 7),              # E-F, E-H
    (2, 5, 6), (8, 5, 7),              # F-G, F-H
    (5, 6, 7)                          # G-H
]
n = 8

adj = {u: [] for u in range(n)}
for w, u, v in edges:
    adj[u].append((w, v))
    adj[v].append((w, u))


class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        # сжатие путей
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        # объединение по рангу
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        return True


def kruskal(n, edges):
    # edges: [(weight, u, v), ...]
    edges = sorted(edges)
    dsu = DSU(n)
    mst = []
    total = 0
    for w, u, v in edges:
        if dsu.union(u, v):
            mst.append((u, v, w))
            total += w
            if len(mst) == n - 1:
                break
    return mst, total


def prim(n, adj, start=0):
    # adj: {u: [(weight, v), ...]}
    visited = [False] * n
    mst = []
    total = 0
    pq = [(0, start, -1)]  # (weight, vertex, parent)
    while pq and len(mst) < n - 1:
        w, u, p = heapq.heappop(pq)
        if visited[u]:
            continue
        visited[u] = True
        if p != -1:
            mst.append((p, u, w))
            total += w
        for weight, v in adj[u]:
            if not visited[v]:
                heapq.heappush(pq, (weight, v, u))
    return mst, total


def cut_for_edge(mst, n, a, b):
    # убираем ребро a-b из дерева, вершины, достижимые из a, это множество S
    tree = {u: [] for u in range(n)}
    for u, v, w in mst:
        if {u, v} != {a, b}:
            tree[u].append(v)
            tree[v].append(u)
    S = {a}
    stack = [a]
    while stack:
        x = stack.pop()
        for y in tree[x]:
            if y not in S:
                S.add(y)
                stack.append(y)
    return S

In [2]:
mst_k, total_k = kruskal(n, edges)
print('Краскал:')
for u, v, w in mst_k:
    print('  ', names[u] + '-' + names[v], w)
print('  суммарный вес:', total_k)

mst_p, total_p = prim(n, adj, 0)
print('Прим (старт A):')
for u, v, w in mst_p:
    print('  ', names[u] + '-' + names[v], w)
print('  суммарный вес:', total_p)
print('Веса совпадают:', total_k == total_p)

Краскал:
   B-C 2
   F-G 2
   C-D 3
   E-H 3
   A-C 4
   E-F 4
   B-F 5
  суммарный вес: 23
Прим (старт A):
   A-C 4
   C-B 2
   C-D 3
   D-E 5
   E-H 3
   E-F 4
   F-G 2
  суммарный вес: 23
Веса совпадают: True


In [3]:
# разрез для ребра E-F из MST
S = cut_for_edge(mst_k, n, 4, 5)
rest = [names[v] for v in range(n) if v not in S]
print('S =', [names[v] for v in sorted(S)], ', остальные вершины:', rest)
cross = [(w, u, v) for w, u, v in edges if (u in S) != (v in S)]
print('Ребра, пересекающие разрез:')
for w, u, v in sorted(cross):
    print('  ', names[u] + '-' + names[v], w)
w, u, v = min(cross)
print('Минимальное ребро разреза:', names[u] + '-' + names[v], w)

S = ['E', 'H'] , остальные вершины: ['A', 'B', 'C', 'D', 'F', 'G']
Ребра, пересекающие разрез:
   E-F 4
   D-E 5
   G-H 5
   B-E 7
   F-H 8
Минимальное ребро разреза: E-F 4
